In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


In [ ]:
import kagglehub
#IMPORTING/LOADING THE OASIS DATA USED IN THE MODEL TRAINING 
# Download latest version
path = kagglehub.dataset_download("ninadaithal/imagesoasis")

print("Path to dataset files:", path)

In [1]:
#HERE THE NECCESSARY LIBRARIES ARE IMPORTED
#TORCH AND CV2 BEING THE DEEP LEARNING AND COMPUTER VISION LIBRARIES RESPECTIVELY

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import cv2 
import os
import torch
from torchvision import datasets
from torch.utils.data import Dataset,DataLoader
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch import nn

if torch.backends.mps.is_available():
    device=torch.device('mps')
elif torch.cuda.is_available():
    device=torch.device('cuda')
print(device)

#SINCE WE ARE USING KAGGLE NIVIDIA GPU T4 X2 FOR OUR COMPUTATION OUR DEVICE SHALL BE CUDA

cuda


In [2]:
#THE OASIS DATASET HAS MRI IN THE FORM OF JPEG SO INORDER TO CONVERT THAT INTO PIL FORMAT WE USE THE RESPECTIVE LIBRARY
from PIL import Image

DATASET_PATH = '/kaggle/input/datasets/ninadaithal/imagesoasis/Data'
data = []
labels = []

classes = ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']

dataset=[]
labels=[]

#THE IMAGES FROM THE DATASET ARE SEPERATED INTO FILES BASED ON THEIR CATEGORY
#INORDER TO PUT ALL THE MRI IMAGE IN ONE WE USE DATASET AND THEIR LABELS ARE APPENDED IN LABELS
for category in classes:
    path = f'{DATASET_PATH}/{category}'
    for img in os.listdir(path):
        img_path = os.path.join(path, img)
        dataset.append(img_path)
        labels.append(category)
print(f"The amount of MRI images present in the dataset is: {len(dataset)}")


The amount of MRI images present in the dataset is: 86437


In [17]:
#HERE THE MAIN DARTASET CREATING ALONG WITH THE IMAGE PREPROCESSING STEPS ARE HAPPENING
#FIRST THE IMAGES(STILL IN THEIR PATH) ARE CONVERTED INTO A PIL IMAGE FIRST 
#FOR CLAHE AND BILATERAL FILTERING TO OCCUR THE IMAGE MUST IN THE FORM OF A NUMPY ARRAY
#SO WE FIRST CONVER THE IMAGE INTO A ARRAY THEN APPLY CLAHE+BILATERAL FILTER ON IT THEN AGAIN CONVERTG BACK TO AN ARRAY
#IT ALSO HAS THE TRANSFORM PART PRESENT IN IT WHICH CONVERTS THE PIL IMAGE TO TENSORS AND RESIZES IT ALONG WITH NORMALIZING IT


class customDataset(Dataset):
    def __init__(self,img_path,labels,transform=None,image_enhancement=True):
        self.img_paths=list(img_path)
        self.labels=list(labels)
        self.transform=transform
        self.imageEnhance=image_enhancement
    def __len__(self):
        return len(self.img_paths)
        
    def __getitem__(self,idx):
        path=self.img_paths[idx]
        image=Image.open(path).convert("RGB")
        label=self.labels[idx]
        
        if self.imageEnhance:
            image=np.array(image)
            image_bw=cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=2, tileGridSize=(8, 8))
            
            image_bw[:,:,0] = clahe.apply(image_bw[:,:,0]) #APPLYING CLAHE ONLY TO L
            clahe_img = cv2.cvtColor(image_bw, cv2.COLOR_LAB2RGB)
            
            final_img=cv2.bilateralFilter(src=clahe_img,d=9,sigmaColor=75,sigmaSpace=75)
            image = Image.fromarray(final_img.astype('uint8'))
            
        if self.transform:
            image=self.transform(image)
        else:
            raise ValueError("Image not converted to tensor.")
        
            
        return image,label

In [25]:
#ENCODING THE LABELS INTO 0,1,2,3 FOR EACH CLASS PRESENT IN THE DATASET

from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
labels_encoded=le.fit_transform(labels)

In [26]:
#THE DATASET IS SPLIT INTO THE TRAINING DATASETS AND TESTING DATASETS

X_train,X_test,Y_train,Y_test=train_test_split(dataset,labels_encoded,random_state=42,test_size=0.2,shuffle=True)
len(Y_test)

17288

In [27]:
#TRANSFORMATION FOR TRAINING AND VALIDATION DATA IS DENOTED HERE

train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4916,0.4498,0.4000],std=[0.2474,0.2362,0.2322])
])
val_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4916,0.4498,0.4000],std=[0.2474,0.2362,0.2322])
])

In [28]:
#CUSTOMDATASET IS CALLED FOR IMAGE PREPROCESSING THE ENTIRE TRAINING AND VALIDATION DATSET IS PREPROCESSED HERE
train_dataset=customDataset(X_train,Y_train,transform=train_transform,image_enhancement=True)
val_dataset=customDataset(X_test,Y_test,transform=val_transform,image_enhancement=True)

In [29]:
#DATALOADER IS USED FOR LOADING THE DATA INTO THE GPU AND THE TRAINING LOOP
#AND VALIDATION EASILY

train_loader = DataLoader(
    train_dataset,
    batch_size=8, 
    shuffle=True,
    num_workers=2, 
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


In [51]:
#THE MAIN ARCHITECTURE IF 3 LAYERED CNN MODEL 

class modelCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(modelCNN, self).__init__()
        
        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.layer3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        ) 
        self.flatten_size = self._get_flatten_size()
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flatten_size, 256),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def _get_flatten_size(self):
        
        dummy = torch.randn(1, 3, 224, 224)
        x = self.layer1(dummy)
        x = self.layer2(x)
        x = self.layer3(x)
        return x.view(1, -1).shape[1]

        self.fc_layers = nn.Sequential(
            nn.Linear(self.flatten_size, 256),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):  
        x = self.layer1(x)  
        x = self.layer2(x)
        x = self.layer3(x)
        x = x.view(x.size(0), -1)  
        x = self.fc_layers(x)
        return x

model = modelCNN().to(device)

In [58]:
EPOCH=8
learning_rate=0.001

#OPTIMIZER AND LOSS FUNCTION

optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)
loss_function=nn.CrossEntropyLoss()
total_loss=0

#TRAINING LOOP
for epoch in range(EPOCH):
    for idx,(image,label) in enumerate(train_loader):
        image=image.to(device)
        label=label.to(device)
        #FOWARD PASS
        outputs=model(image)
        loss=loss_function(outputs,label)

        #BACKWARD PASS
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if (idx + 1) % 3000== 0:
            print(f'Epoch [{epoch+1}/{EPOCH}], Step [{idx+1}], Loss: {loss.item():.4f}')

Epoch [1/8], Step [3000], Loss: 0.1982
Epoch [1/8], Step [6000], Loss: 0.1864
Epoch [2/8], Step [3000], Loss: 0.0516
Epoch [2/8], Step [6000], Loss: 0.0137
Epoch [3/8], Step [3000], Loss: 0.0058
Epoch [3/8], Step [6000], Loss: 0.0107
Epoch [4/8], Step [3000], Loss: 0.0267
Epoch [4/8], Step [6000], Loss: 0.0003
Epoch [5/8], Step [3000], Loss: 0.0008
Epoch [5/8], Step [6000], Loss: 0.0246
Epoch [6/8], Step [3000], Loss: 0.0035
Epoch [6/8], Step [6000], Loss: 0.0031
Epoch [7/8], Step [3000], Loss: 0.0124
Epoch [7/8], Step [6000], Loss: 0.0016
Epoch [8/8], Step [3000], Loss: 0.0001
Epoch [8/8], Step [6000], Loss: 0.0006


In [68]:
#AFTER RAINING THE MODEL AND RUNNING THE MODEL THROUGH THE VALIDATION SET THE MODEL HAS ACHIEVED
#AN ACCURACY OF 99.9%

import torchmetrics
acc = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
model.eval()
model.to(device)  

with torch.no_grad():
    for images, labels in val_loader:
       
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        acc(preds, labels)

test_accuracy = acc.compute()
print(f"Test accuracy: {test_accuracy}")

Test accuracy: 0.9998264908790588
